## Imports

In [1]:
import pandas as pd
import openai

## Read Intervention Data

In [2]:
df_interventions = pd.read_csv("../data/interventions.csv")
df_interventions.head()

KeyboardInterrupt: 

In [ ]:
def validate_fault_code(fault_code):
    if isinstance(fault_code, str) and len(fault_code.split('-')[0]) == 1 and len(fault_code.split('-')[1]) == 3:
        return True
    return False

In [ ]:
df_interventions['is_valid_fault_code'] = df_interventions['fault_code'].apply(validate_fault_code)

## Preprocessing for Vector Store

In [ ]:
df_interventions_cm = df_interventions[df_interventions["intervention_type"] == "CM"]

In [ ]:
def create_summary(row):
    summary_parts = []
    
    if row.get('is_valid_fault_code') is True:
        summary_parts.append(f"[FAULT_CODE] {row['fault_code']}")
    
    if pd.notna(row.get('related_intervention')):
        summary_parts.append(f"[RELATED_INTERVENTION] {row['related_intervention']}")
        
    
    summary_parts.append(f"[EVENT] {row['events']}")
    summary_parts.append(f"[COMMENTS] {row['comments']}")
    
    return "\n".join(summary_parts).strip()


In [ ]:
df_interventions_cm["summary"] = df_interventions_cm.apply(create_summary, axis=1)

/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_50401/1773380293.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interventions_cm["summary"] = df_interventions_cm.apply(create_summary, axis=1)


## Full Load

In [ ]:
df_interventions_cm_sample = df_interventions_cm.copy()

In [ ]:
columns_to_keep = ["id", "date_start", "machine", "duration_min", "summary"]

data_to_embed = df_interventions_cm_sample[columns_to_keep].to_dict(orient='records')

## Create Qdrant Collection

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, SparseVectorParams, Document, Modifier, PayloadSchemaType

In [ ]:
qdrant_host = "http://localhost:6333"

In [ ]:
def create_or_get_collection(client, collection_name, vector_size):
    try:
        client.get_collection(collection_name)
        print(f"Collection '{collection_name}' already exists.")
    except Exception as e:
        print(f"Creating collection '{collection_name}'...")
        client.recreate_collection(
            collection_name=collection_name,
            vectors_config={"text-embedding-3-small": VectorParams(size=vector_size, distance=Distance.COSINE)},
            sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)}
        )

In [ ]:
collection_name = "cm_interventions_hybrid"

embedding_model = "text-embedding-3-small"

embedding_size = 1536

In [ ]:
qdrant_client = QdrantClient(qdrant_host)

create_or_get_collection(qdrant_client, collection_name, vector_size=embedding_size)

Creating collection 'cm_interventions_hybrid'...


/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_50401/538891875.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [ ]:
qdrant_client.create_payload_index(
    collection_name=collection_name,
    field_name="summary",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

## Batch Upsert

In [ ]:
def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

def embed_text_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1

    return all_embeddings

In [ ]:
import uuid

In [ ]:
data_to_embed

[{'id': 'INT-2022-0002',
  'date_start': '2022-01-01 07:24',
  'machine': 'CB-200',
  'duration_min': 82,
  'summary': '[FAULT_CODE] B-006\n[EVENT] Carry roller bearing alarm B-006. Elevated vibration and audible grinding noise on roller section 36.\n[COMMENTS] Fault B-006: roller #36 bearing failure. Vibration 19.0 mm/s RMS (threshold 4.5 mm/s). Root cause: encoder signal cable intermittent connection at connector. Full LOTO. Component replaced. All related systems checked for secondary damage — none found. Roller replaced. Section 36 vibration nominal post-repair. Component flagged for replacement at next planned downtime.'},
 {'id': 'INT-2022-0004',
  'date_start': '2022-01-03 06:14',
  'machine': 'IH-300',
  'duration_min': 196,
  'summary': '[FAULT_CODE] H-004\n[EVENT] Frequency deviation alarm H-004. RF output frequency drifted to 36 kHz (nominal 8 kHz).\n[COMMENTS] Fault H-004: frequency deviation 0.07% from nominal. Root cause: coupling misalignment following recent maintenance

In [ ]:
batch = [d['summary'] for d in data_to_embed]

In [ ]:
embeddings = embed_text_batch(batch, model=embedding_model, batch_size=100)

Processed 101 of 2504
Processed 102 of 2504
Processed 103 of 2504
Processed 104 of 2504
Processed 105 of 2504
Processed 106 of 2504
Processed 107 of 2504
Processed 108 of 2504
Processed 109 of 2504
Processed 110 of 2504
Processed 111 of 2504
Processed 112 of 2504
Processed 113 of 2504
Processed 114 of 2504
Processed 115 of 2504
Processed 116 of 2504
Processed 117 of 2504
Processed 118 of 2504
Processed 119 of 2504
Processed 120 of 2504
Processed 121 of 2504
Processed 122 of 2504
Processed 123 of 2504
Processed 124 of 2504
Processed 125 of 2504
Processed 126 of 2504


In [ ]:
def upsert_points(client, collection_name, records, dense_embeddings, metadata, batch_size=100):
    points = []
    for record, dense in zip(records, dense_embeddings):
        point = PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "text-embedding-3-small": dense,
                "bm25": Document(text=record['summary'], model="qdrant/bm25")
            },
            payload={
                "id": record['id'],
                "date_start": record['date_start'],
                "machine": record['machine'],
                "duration_min": record['duration_min'],
                "summary": record['summary'],
                "embedding_model": metadata.get("embedding_model", ""),
                "created_at": metadata.get("created_at", "")
            }
        )
        points.append(point)
    
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        client.upsert(collection_name=collection_name, points=batch)
        print(f"Upserted batch {i // batch_size + 1}/{(len(points) + batch_size - 1) // batch_size} ({len(batch)} points).")
    
    print(f"Done. Total {len(points)} points upserted into '{collection_name}'.")

In [ ]:
metadata_info = {"embedding_model": embedding_model, "created_at": pd.Timestamp.now().isoformat()}

In [ ]:
upsert_points(qdrant_client, collection_name, data_to_embed, embeddings, metadata_info, batch_size=100)

Upserted batch 1/26 (100 points).
Upserted batch 2/26 (100 points).
Upserted batch 3/26 (100 points).
Upserted batch 4/26 (100 points).
Upserted batch 5/26 (100 points).
Upserted batch 6/26 (100 points).
Upserted batch 7/26 (100 points).
Upserted batch 8/26 (100 points).
Upserted batch 9/26 (100 points).
Upserted batch 10/26 (100 points).
Upserted batch 11/26 (100 points).
Upserted batch 12/26 (100 points).
Upserted batch 13/26 (100 points).
Upserted batch 14/26 (100 points).
Upserted batch 15/26 (100 points).
Upserted batch 16/26 (100 points).
Upserted batch 17/26 (100 points).
Upserted batch 18/26 (100 points).
Upserted batch 19/26 (100 points).
Upserted batch 20/26 (100 points).
Upserted batch 21/26 (100 points).
Upserted batch 22/26 (100 points).
Upserted batch 23/26 (100 points).
Upserted batch 24/26 (100 points).
Upserted batch 25/26 (100 points).
Upserted batch 26/26 (4 points).
Done. Total 2504 points upserted into 'cm_interventions_hybrid'.


## Hybrid Retrieval

In [ ]:
from qdrant_client.models import Prefetch, RrfQuery, Rrf

In [ ]:
COLLECTION_NAME=collection_name

In [ ]:
def retrieve(query, top_k=5):
    vector = embed_text(query)
    
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=vector,
                using="text-embedding-3-small",
                limit=top_k // 2
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=top_k // 2
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[1, 1])),
        limit=top_k
    ).points


    return hits

In [ ]:
query = "roller issue"
results = retrieve(query, top_k=20)

In [ ]:
results

[ScoredPoint(id='df73d627-4148-461e-a8c1-1c551ddc7245', version=10, score=0.5, payload={'id': 'INT-2023-0139', 'date_start': '2023-02-09 07:53', 'machine': 'CB-200', 'duration_min': 295, 'summary': '[FAULT_CODE] B-006\n[EVENT] Carry roller bearing alarm B-006. Elevated vibration and audible grinding noise on roller section 5.\n[COMMENTS] Fault B-006: roller #5 bearing failure. Vibration 44 mm/s RMS (threshold 4.5 mm/s). Root cause: coolant concentration out of spec — insufficient corrosion inhibitor. Fault recurrence from previous intervention (see related_intervention). Root cause re-investigated. Roller replaced. Section 5 vibration nominal post-repair. Component flagged for replacement at next planned downtime.', 'embedding_model': 'text-embedding-3-small', 'created_at': '2026-04-06T10:07:51.838215'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='c667df92-92c8-4822-8236-acfa4425d340', version=5, score=0.5, payload={'id': 'INT-2022-0518', 'date_start': '2022-07-03 